####  Exercise: Linkage Criteria Comparison & Cophenetic Analysis
In this exercise, you will implement a comparison loop across all four linkage methods (`ward`, `complete`, `average`, `single`), compute their Cophenetic Correlation Coefficients, plot four side-by-side dendrograms, and evaluate Silhouette scores. Complete all `TODO` sections.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, cophenet, dendrogram
from scipy.spatial.distance import pdist
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

## 1. Load Data & Prepare Pairwise Distance Matrix

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist

data_path = Path("dataset/hierarchical_sample_data.csv")
if not data_path.exists():
    data_path = Path("../dataset/hierarchical_sample_data.csv")
if not data_path.exists():
    data_path = Path("Day9/dataset/hierarchical_sample_data.csv")

df = pd.read_csv(data_path)
features = ['feature_1', 'feature_2', 'feature_3', 'feature_4']
X = df[features]

# Scale features
X_scaled = StandardScaler().fit_transform(X)

# TODO 1: Calculate condensed pairwise Euclidean distances using pdist()
pairwise_dists = pdist(X_scaled, metric="euclidean")
print(f"Dataset Shape: {df.shape}")
print(f"Computed pairwise distances for {len(X_scaled)} samples (condensed size: {len(pairwise_dists)})")


Dataset Shape: (150, 5)
Computed pairwise distances for 150 samples (condensed size: 11175)


## 2. Compare Linkage Criteria in a Loop

In [3]:
linkage_methods = ['ward', 'complete', 'average', 'single']
results = {}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, method in enumerate(linkage_methods):
    # TODO 2: Compute hierarchical linkage matrix Z using linkage()
    Z = linkage(X_scaled, method=method)
    
    # TODO 3: Calculate Cophenetic Correlation Coefficient using cophenet()
    coph_corr, _ = cophenet(Z, pairwise_dists)
    
    # TODO 4: Fit AgglomerativeClustering with n_clusters=3 and current linkage method
    # Calculate silhouette score on X_scaled
    clusterer = AgglomerativeClustering(n_clusters=3, linkage=method)
    cluster_labels = clusterer.fit_predict(X_scaled)
    sil_score = silhouette_score(X_scaled, cluster_labels)
    
    results[method] = {'coph_corr': round(coph_corr, 4), 'silhouette': round(sil_score, 4)}
    
    # Plot dendrogram
    dendrogram(Z, ax=axes[idx], labels=df['sample_id'].values, leaf_font_size=8)
    axes[idx].set_title(f"Method: {method.upper()} | Coph Corr: {coph_corr:.3f} | Silhouette: {sil_score:.3f}", fontsize=11)
    axes[idx].set_xlabel("Sample ID", fontsize=9)
    axes[idx].set_ylabel("Distance", fontsize=9)
    axes[idx].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig("linkage_comparison_dendrograms.png", dpi=150)
plt.show()

# Display summary table
summary_df = pd.DataFrame(results).T
print("--- Linkage Comparison Summary ---")
display(summary_df)


--- Linkage Comparison Summary ---


C:\Users\conne\AppData\Local\Temp\ipykernel_20832\935982725.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,coph_corr,silhouette
ward,0.8912,0.8346
complete,0.8899,0.8346
average,0.8929,0.8346
single,0.8837,0.8346
